# 05 - Results Analysis & Discussion

This notebook consolidates all experimental results, provides a final comparison
of all methods, and discusses limitations, ethics, and clinical relevance.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

RESULTS_DIR = Path("../results")

## 1. Load saved results

In [ ]:
result_files = sorted(RESULTS_DIR.glob("*.json"))
print("Found result files:")
for f in result_files:
    print(f"  {f.name}")

all_results = {}
for f in result_files:
    with open(f) as fh:
        all_results[f.stem] = json.load(fh)

## 2. Final Model Comparison

We compare all four methods on the held-out test set (624 images: 234 Normal, 390 Pneumonia).

In [ ]:
comparison_metrics = ["auroc", "f1_macro", "sensitivity", "specificity", "precision", "npv", "accuracy"]

model_names = [
    "Logistic Regression",
    "Shallow CNN",
    "ResNet-18 Fine-tune",
    "DenseNet-Attention",
]

baseline_key = [k for k in all_results if "baseline" in k]
model_key = [k for k in all_results if "model_result" in k]

if baseline_key and model_key:
    bl = all_results[baseline_key[0]]
    ml = all_results[model_key[0]]

    results_map = {
        "Logistic Regression": bl.get("logistic_regression", {}),
        "Shallow CNN": bl.get("shallow_cnn", {}),
        "ResNet-18 Fine-tune": ml.get("resnet18", {}),
        "DenseNet-Attention": ml.get("densenet_attention", {}),
    }

    rows = []
    for name in model_names:
        m = results_map.get(name, {})
        row = {"Model": name}
        for metric in comparison_metrics:
            row[metric.upper()] = f"{m.get(metric, 0):.4f}"
        rows.append(row)

    comparison_df = pd.DataFrame(rows)
    print(comparison_df.to_string(index=False))
else:
    print("Result files not found. Run notebooks 02 and 03 first.")

In [ ]:
if baseline_key and model_key:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    primary_metrics = ["auroc", "f1_macro", "sensitivity", "specificity"]
    x = np.arange(len(primary_metrics))
    width = 0.18

    for i, name in enumerate(model_names):
        m = results_map[name]
        values = [float(m.get(metric, 0)) for metric in primary_metrics]
        offset = (i - len(model_names) / 2 + 0.5) * width
        axes[0].bar(x + offset, values, width, label=name)

    axes[0].set_xticks(x)
    axes[0].set_xticklabels([m.upper() for m in primary_metrics])
    axes[0].set_ylabel("Score")
    axes[0].set_title("Primary Metrics Comparison")
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0.2, 1.05)
    axes[0].grid(True, alpha=0.3, axis="y")

    clinical_metrics = ["sensitivity", "npv", "precision"]
    x2 = np.arange(len(clinical_metrics))

    for i, name in enumerate(model_names):
        m = results_map[name]
        values = [float(m.get(metric, 0)) for metric in clinical_metrics]
        offset = (i - len(model_names) / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name)

    axes[1].set_xticks(x2)
    axes[1].set_xticklabels([m.upper() for m in clinical_metrics])
    axes[1].set_ylabel("Score")
    axes[1].set_title("Clinical Metrics Comparison")
    axes[1].legend(fontsize=8)
    axes[1].set_ylim(0.5, 1.05)
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig("../results/final_comparison.png", bbox_inches="tight")
    plt.show()

## 3. Ablation Study Summary

In [ ]:
ablation_key = [k for k in all_results if "ablation" in k]

if ablation_key:
    abl = all_results[ablation_key[0]]

    if "augmentation_ablation" in abl:
        print("Augmentation ablation:")
        for level, metrics in abl["augmentation_ablation"].items():
            print(f"  {level:>10s}: AUROC={metrics['auroc']:.4f}, "
                  f"F1={metrics['f1_macro']:.4f}, "
                  f"Sens={metrics['sensitivity']:.4f}, "
                  f"Spec={metrics['specificity']:.4f}")

    if "data_size_ablation" in abl:
        print("\nData size ablation:")
        for frac, metrics in abl["data_size_ablation"].items():
            print(f"  {float(frac)*100:5.0f}%: AUROC={metrics['auroc']:.4f}, "
                  f"F1={metrics['f1_macro']:.4f}")

    if "component_ablation" in abl:
        print("\nComponent ablation:")
        for name, metrics in abl["component_ablation"].items():
            print(f"  {name:>35s}: AUROC={metrics['auroc']:.4f}, "
                  f"F1={metrics['f1_macro']:.4f}")

    if "multi_run_summary" in abl:
        print("\nMulti-run variability (DenseNet-Attention, 5 seeds):")
        for metric, stats in abl["multi_run_summary"].items():
            print(f"  {metric:>15s}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
else:
    print("Ablation results not found. Run notebook 04 first.")

## 4. Discussion

### Key Findings

**1. Transfer learning is the dominant factor for performance.**

The clearest result across all experiments is the gap between models with and without
ImageNet pretraining. The Logistic Regression baseline (AUROC 0.897) and Shallow CNN
(AUROC 0.912) both lag well behind ResNet-18 (AUROC 0.986) and DenseNet-Attention
(AUROC 0.986). Both transfer learning models achieve near-identical AUROC, suggesting
that the pretrained feature representations are already highly effective for this task
and the architecture choice matters less than the pretraining itself.

**2. DenseNet-Attention achieves the best balance across all metrics.**

While AUROC is comparable between the two deep models, DenseNet-Attention outperforms
ResNet-18 on F1 (0.927 vs. 0.895), specificity (0.833 vs. 0.761), and precision
(0.909 vs. 0.874). Both models achieve very high sensitivity (>0.99) and NPV (>0.98),
which is critical for a screening application. The specificity gap means DenseNet-Attention
produces fewer false alarms, reducing unnecessary follow-up burden.

**3. The baselines reveal different failure modes.**

Logistic Regression has very high sensitivity (0.987) but extremely low specificity (0.376),
meaning it flags almost everything as pneumonia. It functions effectively as a majority-class
predictor with slight refinement. The Shallow CNN improves specificity to 0.603 but at the
cost of lower sensitivity (0.941). This trade-off is expected: without rich pretrained
features, the model struggles to simultaneously capture both classes well.

**4. Standard augmentation is optimal; both "none" and "basic" overfit.**

The augmentation ablation reveals a clear pattern. With no augmentation, training loss
drops to near-zero (strong overfitting) and test specificity collapses to 0.462. Basic
augmentation (just flips and small rotation) performs even worse on specificity (0.261),
likely because the mild augmentation is not enough to prevent overfitting but adds noise
that disrupts the normal-class boundary. Standard augmentation yields the best test
performance (AUROC 0.989, F1 0.883), and heavy augmentation slightly degrades results
(AUROC 0.984, F1 0.838), possibly because elastic transforms and coarse dropout distort
the X-ray features too aggressively at 224x224 resolution.

**5. The model is robust to reduced training data.**

Thanks to transfer learning, even 25% of the training data (approximately 1,100 images)
yields AUROC 0.984 and sensitivity 0.997. The main degradation is in specificity, which
drops from 0.697 (100%) to 0.624 (25%). This is encouraging for real-world settings
where labeled medical data is scarce.

**6. The component ablation yields a surprising result.**

Contrary to our hypothesis, removing the attention module or focal loss does not
consistently degrade performance. The "No attention" configuration achieved AUROC 0.989
and F1 0.901, slightly higher than the full model (AUROC 0.982, F1 0.891). This suggests
that for this particular dataset and task, the DenseNet-121 backbone already provides
sufficiently discriminative features, and the attention mechanism adds marginal value.
The focal loss shows a clearer contribution: removing it while keeping attention reduces
F1 from 0.891 to 0.855, indicating it helps with the class imbalance.

**7. Results are reproducible with moderate variability.**

Across 5 random seeds, the DenseNet-Attention model shows AUROC 0.982 +/- 0.007 and
sensitivity 0.996 +/- 0.003. However, specificity shows higher variance at
0.663 +/- 0.118. This indicates the model reliably detects pneumonia cases but
is less consistent in correctly identifying normal cases. The F1 variance of 0.059
is notable and suggests the model's operating point is sensitive to initialization,
particularly for the minority (normal) class.

### Limitations

**Dataset limitations:**

- The dataset comes from a single center (Guangzhou Women and Children's Medical Center).
  Models trained on single-center data are known to learn institution-specific artifacts
  (scanner type, positioning protocol, image labeling conventions) rather than purely
  clinical features. External validation on multi-center data would be essential.

- All patients are pediatric (ages 1-5). Pediatric pneumonia has different radiographic
  patterns than adult pneumonia. This model should absolutely not be deployed for adult
  patients without retraining and validation.

- The binary classification groups bacterial and viral pneumonia together. These have
  different treatment pathways (antibiotics vs. supportive care), so a more clinically
  useful system would distinguish between them.

- The original validation split (16 images) was too small, requiring us to re-split.
  This is a common issue with Kaggle datasets that are not designed for rigorous ML research.

**Methodological limitations:**

- The high specificity variance across seeds (0.663 +/- 0.118) is concerning. In a clinical
  setting, the false positive rate needs to be predictable. Further work could investigate
  ensemble methods or calibration techniques to stabilize this.

- Our Grad-CAM visualizations show the model attending to lung regions, but we have not
  verified whether it focuses on the specific pathological patterns (consolidation, ground-glass
  opacity) vs. correlated but non-causal features. A radiologist review of attention maps
  would be needed.

- We did not perform hyperparameter search beyond the default configuration. The training
  was relatively short (15 epochs with early stopping), which may not be optimal for all models.

### Ethical Considerations

**Decision support, not replacement.** This system is designed to assist radiologists, not
replace them. The final diagnosis must always come from a qualified physician who considers
the full clinical picture (symptoms, lab results, patient history) alongside imaging.

**Bias and fairness.** The model is trained exclusively on data from Chinese pediatric patients.
Deploying it for other demographic groups without validation would introduce unquantified risk.
Chest X-ray appearance varies with body habitus, age, and ethnicity. A fair deployment would
require validation across diverse populations.

**Automation bias.** There is a well-documented risk that clinicians may over-rely on AI
predictions, especially when the system has high reported accuracy. Our high sensitivity
(>99%) could create a false sense of security. Clear communication of uncertainty, confidence
calibration, and training on appropriate use are essential.

**Patient consent and data privacy.** The dataset is de-identified and publicly available.
However, any real-world deployment must comply with relevant regulations (HIPAA, GDPR,
China's PIPL) and ensure informed consent for AI-assisted diagnosis.

**Regulatory requirements.** Medical AI systems require regulatory approval before clinical
use (FDA 510(k) or De Novo, EU MDR Class IIa, etc.). This requires prospective clinical
trials and post-market surveillance, which are beyond the scope of this project.

### Clinical Relevance

Despite the above limitations, the results demonstrate potential clinical value in two scenarios:

**Triage in resource-limited settings.** With sensitivity above 99% and NPV above 98%, the
system could reliably flag cases that need urgent radiologist attention. In settings with
limited radiology staffing, this could reduce diagnostic delays for pneumonia.

**Quality assurance as a second reader.** The model could serve as an automated second reader,
catching potential misses in high-volume reading environments. Its high sensitivity makes it
particularly suitable for this role, where the cost of missing a positive case outweighs
the cost of occasional false alarms.

In [ ]:
print("Analysis complete.")
print("All results and figures are saved in the results/ directory.")